In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.parent.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from scripts.sweep_analysis import (
    load_sweep_summary,
    bootstrap_sweep_raw_parquets,
    summarize_bootstrap,
    build_latency_delta_panel_data,
    plot_latency_delta_panel_data,
)
from src.config import CONFIG

In [ ]:
OUTPUT_ROOT = PROJECT_DIR / "reports/raw_latency_sweep"

TICKER = "CRM" #AMGN CRM* CSCO* GILD ISRG* LCID* PEP RXRX SBUX WMT*   PEP, RXRX, SBUX, WMT median plot looks weird
MODEL_TYPE = "gru"
LIFECYCLE_AWARE = True

# CRM, CSCO, ISRG, LCID, WMT

def clean_sweep(df):
    cols = [
        "latency_ms",
        "mean_is_bps",
        "median_is_bps",
        "time_weighted_mean_is_bps",
        "fill_rate",
        "submitted",
        "filled",
        "unfilled",
        "canceled",
    ]
    return df[[c for c in cols if c in df.columns]].sort_values("latency_ms")

summary = load_sweep_summary(
    OUTPUT_ROOT,
    ticker=TICKER,
    model_type=MODEL_TYPE,
    lifecycle_aware=LIFECYCLE_AWARE,
)

clean_sweep(summary).head()

In [ ]:
plot_df = clean_sweep(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

axes[0].plot(plot_df["latency_ms"], plot_df["median_is_bps"], marker="o")
axes[0].set_title(f"{TICKER} {MODEL_TYPE}: median IS")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("median_is_bps")
axes[0].grid(True, alpha=0.3)

axes[1].plot(plot_df["latency_ms"], plot_df["time_weighted_mean_is_bps"], marker="o")
axes[1].set_title(f"{TICKER} {MODEL_TYPE}: time-weighted mean IS")
axes[1].set_xlabel("Latency (ms)")
axes[1].set_ylabel("time_weighted_mean_is_bps")
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xscale("symlog", linthresh=0.01)

plt.tight_layout()
plt.show()

In [ ]:
SAMPLE_SIZE = None      # None = same size as each raw parquet's metric-ok rows
N_TRIALS = 1000
MIN_HORIZON_MS = 0.001

boot = bootstrap_sweep_raw_parquets(
    OUTPUT_ROOT,
    ticker=TICKER,
    model_type=MODEL_TYPE,
    lifecycle_aware=LIFECYCLE_AWARE,
    sample_size=SAMPLE_SIZE,
    n_trials=N_TRIALS,
    min_horizon_ms=MIN_HORIZON_MS,
    random_state=CONFIG.random_seed,
)

boot_summary = summarize_bootstrap(boot)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for ax, metric, title in [
    (axes[0], "median_is_bps", "Bootstrap median IS"),
    (axes[1], "time_weighted_mean_is_bps", "Bootstrap time-weighted mean IS"),
]:
    x = boot_summary["latency_ms"]
    y = boot_summary[f"{metric}_mean"]
    lo = boot_summary[f"{metric}_ci_low"]
    hi = boot_summary[f"{metric}_ci_high"]

    ax.plot(x, y, marker="o")
    ax.fill_between(x, lo, hi, alpha=0.2)
    ax.set_title(f"{TICKER} {MODEL_TYPE}: {title}")
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel(metric)
    ax.set_xscale("symlog", linthresh=0.01)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

boot_summary

In [ ]:
# Compact paper-style latency sweep figure: 4 panels, one model type per panel.
# Each faint line is one ticker's bootstrap median-IS delta versus 0 ms latency;
# the black line is the cross-ticker average within that panel.

PANEL_SPECS = [
    {
        "title": "GRU",
        "model_type": "gru",
        "tickers": ["CRM", "CSCO", "ISRG", "LCID", "WMT"],
    },
    {
        "title": "Transformer",
        "model_type": "transformer",
        "tickers": ["CRM", "CSCO", "ISRG", "LCID", "WMT"],
    },
    {
        "title": "GRU Transformer",
        "model_type": "gru_transformer",
        "tickers": ["CRM", "CSCO", "ISRG", "LCID", "WMT"],
    },
    {
        "title": "Mamba",
        "model_type": "mamba",
        "tickers": ["CRM", "CSCO", "ISRG", "LCID", "WMT"],
    },
]

PAPER_SAMPLE_SIZE = None
PAPER_N_TRIALS = 1000
PAPER_MIN_HORIZON_MS = 0.001

latency_delta_df = build_latency_delta_panel_data(
    PANEL_SPECS,
    output_root=OUTPUT_ROOT,
    lifecycle_aware=LIFECYCLE_AWARE,
    metric="toxic_cost_mean_bps",
    baseline_latency_ms=0.0,
    sample_size=PAPER_SAMPLE_SIZE,
    n_trials=PAPER_N_TRIALS,
    min_horizon_ms=PAPER_MIN_HORIZON_MS,
    random_state=CONFIG.random_seed,
    progress=True,
)

latency_delta_df.head()

In [ ]:
# TITLE_OVERRIDES = {
#     0: "Mamba / Large-Cap",
#     1: "GRU-Trans. / Large-Cap",
#     2: "Mamba / High-Toxicity",
#     3: "GRU-Trans. / High-Toxicity",
# }

# plot_df = latency_delta_df.copy()
# for panel_idx, title in TITLE_OVERRIDES.items():
#     plot_df.loc[plot_df["panel_index"].eq(panel_idx), "panel_title"] = title

# Pure plotting cell. You can rerun and adjust visual settings without rerunning bootstrap.
PLOT_METRIC = "toxic_cost_mean_bps"
# Options:
# "mean_is_bps"
# "median_is_bps"
# "time_weighted_mean_is_bps"

fig, axes = plot_latency_delta_panel_data(
    latency_delta_df,
    metric=PLOT_METRIC,
    figsize=(10.5, 7.2),
    background_alpha=0.28,
    average_color="black",
    linthresh=0.01,
    sharey=True,
    common_y_limits=True,
    y_limits=None,  # e.g. (-1.0, 1.0) for a manual common range
    legend_y=0.94,
    layout_top=0.90,
)

fig.suptitle("Latency sensitivity: bootstrap toxic cost delta versus 0 ms", y=0.99)
plt.show()

In [ ]:
latency_delta_df.to_csv("latency_delta_bootstrap_results.csv")